In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torchvision.models import EfficientNet_V2_S_Weights
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import pandas as pd
import os
from tqdm import tqdm
import copy
import cv2
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
test_image_dir = "/kaggle/input/cassava-leaf-disease-classification/test_images"

In [ ]:
test_df = pd.read_csv('../input/cassava-leaf-disease-classification/sample_submission.csv')
test_df.head()

In [ ]:
resnet_transforms = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.Resize(224, 224),  # Resize to match ResNet input size
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])
efficientnet_transforms = A.Compose([
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
    A.Resize(384, 384),  # Resize to match EfficientNetV2-S input size
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
class CassavaTestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform_resnet=None, transform_efficientnet=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform_resnet = transform_resnet
        self.transform_efficientnet = transform_efficientnet

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx, 0]  # Image ID
        img_path = os.path.join(self.image_dir, img_name)

        # Load the image using OpenCV
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Apply transformations for ResNet
        if self.transform_resnet:
            augmented_resnet = self.transform_resnet(image=image)
            image_resnet = augmented_resnet['image']
        else:
            image_resnet = None

        # Apply transformations for EfficientNet
        if self.transform_efficientnet:
            augmented_efficientnet = self.transform_efficientnet(image=image)
            image_efficientnet = augmented_efficientnet['image']
        else:
            image_efficientnet = None

        return image_resnet, image_efficientnet, img_name

In [ ]:
test_dataset = CassavaTestDataset(
    test_df,
    test_image_dir,
    transform_resnet=resnet_transforms,
    transform_efficientnet=efficientnet_transforms
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
resnet_model = models.resnet50(pretrained=False)
num_ftrs = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_ftrs, 5)

# Load your trained model weights
resnet_model.load_state_dict(torch.load("/kaggle/input/casava-aug/pytorch/default/1/cassava_leaf_best_model_fine_aug.pth"))
resnet_model = resnet_model.to(device)

# Set the model to evaluation mode
resnet_model.eval()

In [ ]:
efficientnet_model = models.efficientnet_v2_s(weights=None)
num_features_efficientnet = efficientnet_model.classifier[1].in_features
efficientnet_model.classifier[1] = nn.Linear(num_features_efficientnet, 5)

# Load your custom weights
efficientnet_model.load_state_dict(torch.load("/kaggle/input/eff-t/pytorch/default/1/Eff.pth", map_location=device))
efficientnet_model = efficientnet_model.to(device)
efficientnet_model.eval()

In [ ]:


ensemble_predictions = []
image_names = []

with torch.no_grad():
    for images_resnet, images_efficientnet, img_names in test_loader:
        images_resnet = images_resnet.to(device)
        images_efficientnet = images_efficientnet.to(device)

        # Get predictions from ResNet
        outputs_resnet = resnet_model(images_resnet)
        probs_resnet = F.softmax(outputs_resnet, dim=1)

        # Get predictions from EfficientNet
        outputs_efficientnet = efficientnet_model(images_efficientnet)
        probs_efficientnet = F.softmax(outputs_efficientnet, dim=1)

        # Combine predictions (e.g., average)
        combined_probs = (probs_resnet + probs_efficientnet) / 2

        # Get the predicted class
        preds = combined_probs.argmax(dim=1).cpu().numpy()

        ensemble_predictions.extend(preds)
        image_names.extend(img_names)

In [ ]:
submission_df = pd.DataFrame({
    'image_id': image_names,         # Image IDs from the test set
    'label': ensemble_predictions     # Predicted labels for each image
})

# Save the DataFrame as a CSV file without the index
submission_df.to_csv('submission.csv', index=False)

print("Submission file saved as 'submission.csv'")